<a href="https://colab.research.google.com/github/kaplanali11/nlp-fact-checked-summarization/blob/feature%2Fsummarizer-t5/notebooks/02_t5_zero_shot_k3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q transformers datasets sentencepiece accelerate evaluate protobuf

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan Cihaz: {device}")

# Çalışan hazır Türkçe özetleme modeli
model_name = "ozcangundes/mt5-small-turkish-summarization"

print("Model ve tokenizer indiriliyor...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("Model başarıyla yüklendi!")

Kullanılan Cihaz: cuda
Model ve tokenizer indiriliyor...


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (3 outputs left out)

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Model başarıyla yüklendi!


In [5]:
def summarize(text, max_len=128, min_len=20):
    clean_text = text.strip()
    # mt5-small modeli doğrudan metni alabilir
    inputs = tokenizer(
        clean_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            min_length=min_len,
            num_beams=4,
            no_repeat_ngram_size=2,
            early_stopping=True
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test Metni (Resmi Duyuru - Kısıt ve Şart İçeren)
test_metni = """
Dokuz Eylül Üniversitesi Rektörlüğü, Tınaztepe Yerleşkesi bünyesinde ileri teknoloji
ve yapay zeka araştırma merkezinin temellerinin atıldığını duyurdu. Merkezin
inşaatının önümüzdeki ay başlayacağı ve 2027 yılı sonunda tamamlanarak faaliyete
geçeceği bildirildi. Proje kapsamında lisansüstü araştırmacılara ve mühendislik
öğrencilerine özel laboratuvar tahsisi yapılacağı açıklandı. Açılış protokolüne
yalnızca mühendislik fakültesi öğretim üyeleri ve sanayi temsilcileri davet edildi.
"""

print("\n--- ÖRNEK GİRDİ ---")
print(test_metni.strip())

print("\n--- MODELİN ÜRETTİĞİ SIFIR VURUŞ ÖZET ---")
print(summarize(test_metni))


--- ÖRNEK GİRDİ ---
Dokuz Eylül Üniversitesi Rektörlüğü, Tınaztepe Yerleşkesi bünyesinde ileri teknoloji 
ve yapay zeka araştırma merkezinin temellerinin atıldığını duyurdu. Merkezin 
inşaatının önümüzdeki ay başlayacağı ve 2027 yılı sonunda tamamlanarak faaliyete 
geçeceği bildirildi. Proje kapsamında lisansüstü araştırmacılara ve mühendislik 
öğrencilerine özel laboratuvar tahsisi yapılacağı açıklandı. Açılış protokolüne 
yalnızca mühendislik fakültesi öğretim üyeleri ve sanayi temsilcileri davet edildi.

--- MODELİN ÜRETTİĞİ SIFIR VURUŞ ÖZET ---
Dokuz Eylül Üniversitesi , Tınaztepe Yerleşkesi bünyesinde ileri teknoloji ve yapay zeka araştırma merkezinin temellerinin atıldığını duyurdu .


In [6]:
# Test 2: Teknoloji / Olay Haberi
haber_metni = """
Türkiye Uzay Ajansı ve TÜBİTAK UZAY ortaklığıyla yürütülen yeni yerli gözlem uydusu
projesinde kritik tasarım aşaması tamamlandı. Yüksek çözünürlüklü optik kameraya sahip
olan uydunun, tarımsal rekolte tahminleri ve afet yönetiminde anlık veri sağlayacağı
belirtildi. Yetkililer, fırlatmanın 2026 yılının son çeyreğinde gerçekleştirileceğini
ve ana yer kontrol istasyonunun Ankara'da kurulduğunu ifade etti.
"""

print("\n--- HABER METNİ ÖZETİ ---")
print(summarize(haber_metni))

# Test 3: Sayısal / Finans Verisi
ekonomi_metni = """
Merkez Bankası Para Politikası Kurulu, politika faizini 250 baz puan artırarak
yüzde 45 seviyesine çıkardığını açıkladı. Karar metninde, aylık enflasyonun ana
eğiliminde belirgin bir düşüş sağlanana kadar parasal sıkılığın korunacağı vurgulandı.
Toplantı sonrasında döviz kurlarında yatay bir seyir izlenirken, borsa endeksi
günü yüzde 1.2 yükselişle kapattı.
"""

print("\n--- EKONOMİ METNİ ÖZETİ ---")
print(summarize(ekonomi_metni))


--- HABER METNİ ÖZETİ ---
Türkiye Uzay Ajansı ve TÜBİTAK UZAY ortaklığıyla yürütülen yeni yerli gözlem uydusu projesinde kritik tasarım aşaması tamamlandı .

--- EKONOMİ METNİ ÖZETİ ---
Merkez Bankası Para Politikası Kurulu , politika faizini 250 baz puan artırarak yüzde 45 seviyesine çıkardığını açıkladı .


In [7]:
def summarize_advanced(text, min_len=45, max_len=130, length_penalty=2.0, num_beams=5):
    inputs = tokenizer(
        text.strip(),
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            min_length=min_len,              # En az 40-45 token yazmaya zorlar
            length_penalty=length_penalty,   # Uzun ve kapsamlı özetleri ödüllendirir
            num_beams=num_beams,             # Olasılık uzayını daha derin tarar
            no_repeat_ngram_size=3,          # Kelime tekrarlarını kırar
            repetition_penalty=2.5,          # Aynı kalıba saplanmasını engeller
            early_stopping=True
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Tekrar Test Et
print("--- ZORLANMIŞ ÖZET 1 (DUYURU) ---")
print(summarize_advanced(test_metni))

print("\n--- ZORLANMIŞ ÖZET 2 (HABER) ---")
print(summarize_advanced(haber_metni))

print("\n--- ZORLANMIŞ ÖZET 3 (EKONOMİ) ---")
print(summarize_advanced(ekonomi_metni))

--- ZORLANMIŞ ÖZET 1 (DUYURU) ---
Dokuz Eylül Üniversitesi Rektörlüğü , Tınaztepe Yerleşkesi bünyesinde ileri teknoloji ve yapay zeka araştırma merkezinin temellerinin atıldığını duyurdu . Proje kapsamında lisansüstü araştırmacılara ve mühendislik öğrencilerine özel laboratuvar tahsisi yapılacağı açıklandı

--- ZORLANMIŞ ÖZET 2 (HABER) ---
Türkiye Uzay Ajansı ve TÜBİTAK UZAY ortaklığıyla yürütülen yeni yerli gözlem uydusu projesinde kritik tasarım aşaması tamamlandı . Yetkililer , fırlatmanın 2026 yılının son çeyreğinde gerçekleştirileceğini açıkladı .

--- ZORLANMIŞ ÖZET 3 (EKONOMİ) ---
Merkez Bankası Para Politikası Kurulu , aylık enflasyonun ana eğiliminde belirgin bir düşüş sağlanana kadar parasal sıkılığın korunacağını açıkladı . Karar metninde döviz kurlarında yatay bir seyir izlenirken borsa endeksi günü yüzde 1.2 yükselişle kapattı .


###Hafta 8'de kendi veri setimizle modeli fine-tune etmemiz ve K4'ün NLI denetçisiyle kapsam kontrolü yapmamız şart